In [ ]:
# Mount google drive to current runtime session to access required files

'''
from google.colab import drive
drive.mount('/content/drive')
'''


"\nfrom google.colab import drive\ndrive.mount('/content/drive')\n"

In [ ]:
# Import spectrogram zip to local storage for efficient loading

'''
import os
import shutil

shutil.copy('/content/drive/MyDrive/6600_project/piano_roll_150014.zip', '/content/piano_roll_150014.zip')

!unzip -q piano_roll_150014.zip -d /content/

!du -sh /content/piano_roll/*
!find /content/piano_roll -type f | wc -l

'''

"\nimport os\nimport shutil\n\nshutil.copy('/content/drive/MyDrive/6600_project/piano_roll_150014.zip', '/content/piano_roll_150014.zip')\n\n!unzip -q piano_roll_150014.zip -d /content/\n\n!du -sh /content/piano_roll/*\n!find /content/piano_roll -type f | wc -l\n\n"

In [ ]:
# Import Libraries
import os
import shutil
import time
import random
import itertools
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/6600_project/lmd_full_metadata.csv', parse_dates=False)

# Filter out missing BPM
df = df[
    df['initial_bpm'].notna() & (df['initial_bpm'] != '') &
    df["duration_s"].between(15,600) &
    df["initial_bpm"].between(30,240) &
    df["n_notes"].between(50, 50_000)
    ]

print(f"Total usable samples: {len(df)}")
print(f"Distribution of BPM:\n{df['initial_bpm'].describe()}")
df.head(3)

Total usable samples: 150017
Distribution of BPM:
count    150017.000000
mean        114.163743
std          33.206190
min          30.000000
25%          91.000000
50%         115.000000
75%         130.000000
max         240.000000
Name: initial_bpm, dtype: float64


,file,filepath,duration_s,resolution,n_instruments,n_notes,n_tempo_changes,initial_bpm,n_key_changes,initial_key,n_time_sig_changes,initial_time_sig,key_name
0,d66652ff61cdea1eda265490600e6e40.mid,../data/raw_data/lmd_full/d/d66652ff61cdea1eda...,176.59,384,11,5244,1,118.0,0,NaN,1,4/4,NaN
1,d1e757857fb459edf3e543c3c8a79c6a.mid,../data/raw_data/lmd_full/d/d1e757857fb459edf3...,219.15,192,10,4974,1,72.0,0,NaN,1,4/4,NaN
2,d597590e6423c93180f56f2854bfb109.mid,../data/raw_data/lmd_full/d/d597590e6423c93180...,176.97,384,5,3965,62,120.0,13,0.0,2,1/4,C major


# Checking for corrupted files

In [ ]:
df['npy_file'] = df['file'].str.replace(r'\.mid$', '.npy', regex=True)

# Build full file paths
data_dir = '/content/piano_roll/'
df['file_path'] = data_dir + df['npy_file']

# Check which files actually exist
df['file_exists'] = df['file_path'].apply(os.path.exists)

print(f"Files found:   {df['file_exists'].sum()} in csv filepath")
print(f"Number of missing file is {(~df['file_exists']).sum()} from the list")

# Keep only existing files
df = df[df['file_exists']]

tqdm.pandas()  # enables progress_apply

df = df[df['file_path'].progress_apply(
    lambda p: np.load(p).shape == (128, 468)
)]

print("")
print(f"Remaining samples after removing different shapes: {len(df)}")

Files found:   150014 in csv filepath
Number of missing file is 3 from the list


100%|██████████| 150014/150014 [00:03<00:00, 37555.02it/s]



Remaining samples after removing different shapes: 150014


In [ ]:
piano_roll_dir = '/content/piano_roll'
files = [f for f in os.listdir(piano_roll_dir) if f.endswith('.npy')]

random_samples = random.sample(files, 3)

for file in random_samples:
    file_path = os.path.join(piano_roll_dir, file)
    sample = np.load(file_path)
    print(f"File: {file}")
    print(f"  Shape: {sample.shape}")
    print(f"  Dtype: {sample.dtype}")
    print(f"  Min: {sample.min():.4f}, Max: {sample.max():.4f}\n")

File: 8391f45ba216e47c4e464859d1db92a8.npy
  Shape: (128, 468)
  Dtype: uint8
  Min: 0.0000, Max: 73.0000

File: dcd725bc11dfb8698202f77d7ea92509.npy
  Shape: (128, 468)
  Dtype: uint8
  Min: 0.0000, Max: 144.0000

File: e37c68fe9411093d152f27372b251493.npy
  Shape: (128, 468)
  Dtype: uint8
  Min: 0.0000, Max: 228.0000



# Start of CNN&GRU Models


In [ ]:
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')

2.10.0+cu128
True
NVIDIA RTX PRO 6000 Blackwell Server Edition


In [ ]:
Path("/content/piano_roll")

METADATA_CSV      = Path("/content/drive/MyDrive/lmd_full_metadata.csv")
DATA_DIR          = Path("/content/piano_roll")
SAVE              = Path("/content/models")
MODEL02_SAVE_PATH = Path("/content/models/gru_bpm_regressor.pt")

# Ensure required directories exist
SAVE.mkdir(parents=True, exist_ok=True)


# ── Data filters ──────────────────────────────────────────────────────────────
TARGET      = 'initial_bpm'
BPM_MIN     = 30
BPM_MAX     = 240
MAX_SAMPLES = None   # set int (e.g. 10_000) for quick dev runs; None = all

# ── Model / training hyperparameters ─────────────────────────────────────────
FS           = 31.25
SEGMENT_LEN  = 468   # actual piano-roll length: 15 s × 31.25 fps = 468 frames
N_PITCH      = 128
BATCH_SIZE   = 1024
LR           = 1e-3
WEIGHT_DECAY = 1e-4
N_EPOCHS     = 10
PATIENCE     = 3
DROPOUT      = 0.5   # shared dropout rate used by all models
SEED         = 66

# ── Default hyperparameters ─────────────────────────────────────────────────
DEFAULT_LR      = 3e-4
DEFAULT_DROPOUT = 0.3
DEFAULT_HIDDEN  = 256
DEFAULT_CNN     = 4
DEFAULT_GRU     = 1
DEFAULT_MLP     = 1

# ── Hyperparameter search grids ───────────────────────────────────────────────
LR_SEARCH       = [1e-3, 5e-4, 3e-4, 1e-4, 1e-5]
DROPOUT_SEARCH  = [0.2, 0.3, 0.4]
GRU_SIZE_SEARCH = [64, 128, 256]
ARCH_CONFIGS = [
    {'num_cnn_layers': 3, 'num_gru_layers': 1, 'num_mlp_layers': 1},
    {'num_cnn_layers': 4, 'num_gru_layers': 1, 'num_mlp_layers': 1},
    {'num_cnn_layers': 5, 'num_gru_layers': 1, 'num_mlp_layers': 1},
    {'num_cnn_layers': 5, 'num_gru_layers': 1, 'num_mlp_layers': 2},
    {'num_cnn_layers': 3, 'num_gru_layers': 2, 'num_mlp_layers': 1},
    {'num_cnn_layers': 4, 'num_gru_layers': 2, 'num_mlp_layers': 1},
    {'num_cnn_layers': 5, 'num_gru_layers': 2, 'num_mlp_layers': 1},
    {'num_cnn_layers': 5, 'num_gru_layers': 2, 'num_mlp_layers': 2},
]

DEVICE = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

Device: cuda


In [ ]:
class PianoRollDataset(Dataset):
    def __init__(self, file_paths, labels):
        self.file_paths = file_paths
        self.labels = labels

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        x = np.load(self.file_paths[idx]).astype(np.float32)
        x = x / 127.0                        # normalize velocity 0-127 → 0-1
        x = x[np.newaxis, :, :]              # add channel dim → (1, 128, 468)
        return torch.tensor(x), torch.tensor(self.labels[idx], dtype=torch.float32)

In [ ]:
# Optimized DataLoader with higher num_workers and pin_memory
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=os.cpu_count(), pin_memory=True, prefetch_factor=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=os.cpu_count(), pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=os.cpu_count(), pin_memory=True)

# Training Method (Optimized with Mixed Precision)

In [ ]:
def train_model(model, train_loader, val_loader, bpm_std, epochs=N_EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY, checkpoint_path='checkpoint.pth', device=DEVICE):
    start = time.time()
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Use the modern torch.amp API for mixed precision
    scaler = torch.amp.GradScaler('cuda') if device == 'cuda' else None
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.7)

    best_val_mae = float('inf')
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        running_mae  = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True) # set_to_none is slightly faster than zero_grad()

            if device == 'cuda':
                with torch.amp.autocast('cuda'):
                    outputs = model(images).squeeze()
                    loss = criterion(outputs, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(images).squeeze()
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

            running_loss += loss.item()
            running_mae  += torch.abs(outputs - labels).mean().item()

        # Validation and checkpointing logic remains same...
        model.eval()
        val_mae = 0.0
        with torch.no_grad(), torch.amp.autocast('cuda') if device == 'cuda' else contextlib.nullcontext():
            for images, labels in val_loader:
                images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                outputs = model(images).squeeze()
                val_mae += torch.abs(outputs - labels).mean().item()

        val_mae /= len(val_loader)
        val_mae_bpm = val_mae * bpm_std

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            epochs_no_improve = 0
            torch.save({'model_state_dict': model.state_dict()}, checkpoint_path)
            saved_marker = " ✓ saved"
        else:
            epochs_no_improve += 1
            saved_marker = f" (no improvement {epochs_no_improve}/{PATIENCE})"

        print(f"Epoch {epoch+1:2d} | Loss: {running_loss/len(train_loader):.4f} | Val MAE: {val_mae_bpm:.1f} BPM{saved_marker}")
        scheduler.step()
        if epochs_no_improve >= PATIENCE: break

    print(f"Total training time: {(time.time()-start)/60:.1f} minutes")
    return best_val_mae * bpm_std

# Testing method

In [ ]:
def test_model_and_plot(model, test_loader, bpm_std, bpm_mean, checkpoint_path=None, device=DEVICE):
    if checkpoint_path and os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"Loaded weights from {checkpoint_path} (Epoch {checkpoint.get('epoch', 'unknown')})")
    else:
        print("No checkpoint loaded. Using current model weights.")

    model.eval()
    val_mae = 0.0
    predictions = []
    actuals = []

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Evaluating"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images).squeeze()

            # Convert back to original BPM scale
            pred_bpm = (outputs * bpm_std) + bpm_mean
            actual_bpm = (labels * bpm_std) + bpm_mean

            if outputs.dim() == 0:
                pred_bpm = pred_bpm.unsqueeze(0)
                actual_bpm = actual_bpm.unsqueeze(0)

            predictions.extend(pred_bpm.cpu().numpy())
            actuals.extend(actual_bpm.cpu().numpy())

            val_mae += torch.abs(outputs - labels).mean().item()

    val_mae = val_mae / len(test_loader)
    val_mae_bpm = val_mae * bpm_std

    print(f"\nTest MAE: {val_mae_bpm:.2f} BPM")

    # Plotting Actual vs Predicted (Dot Plot)
    plt.figure(figsize=(8, 6))
    plt.scatter(actuals, predictions, alpha=0.3, color='blue', s=10)

    # Perfect prediction diagonal line
    min_val = min(min(actuals), min(predictions))
    max_val = max(max(actuals), max(predictions))
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')

    plt.xlabel('Actual BPM')
    plt.ylabel('Predicted BPM')
    plt.title('Dot Plot: Actual vs Predicted BPM')
    plt.legend()
    plt.grid(True)
    plt.show()

    return np.array(predictions), np.array(actuals)

# CNN with GRU STRUCTURE

In [ ]:
class GRUSpectrogramCNN(nn.Module):
    def __init__(self, num_outputs, gru_hidden=256, num_gru_layers=1,
                 num_cnn_layers=4, num_mlp_layers=2, dropout=DROPOUT):
        super().__init__()

        # Dynamic CNN feature extractor: channels 1→32→64→128→256→512
        cnn_channels = [1, 32, 64, 128, 256, 512]
        layers = []
        for i in range(num_cnn_layers):
            layers += [
                nn.Conv2d(cnn_channels[i], cnn_channels[i+1], kernel_size=3, padding=1),
                nn.BatchNorm2d(cnn_channels[i+1]),
                nn.ReLU(),
                nn.MaxPool2d(2, 2)
            ]
        self.features = nn.Sequential(*layers)

        # GRU input_size is always 2048 regardless of CNN depth:
        # 3 CNN → 128*16=2048, 4 CNN → 256*8=2048, 5 CNN → 512*4=2048
        self.gru = nn.GRU(
            input_size=2048,
            hidden_size=gru_hidden,
            num_layers=num_gru_layers,
            batch_first=True,
            bidirectional=True
        )

        gru_out_size = gru_hidden * 2  # bidirectional

        if num_mlp_layers == 1:
            self.head = nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(gru_out_size, num_outputs)
            )
        else:  # 2-layer MLP
            self.head = nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(gru_out_size, 128),
                nn.ReLU(),
                nn.Linear(128, num_outputs)
            )

    def forward(self, x):
        x = self.features(x)
        B, C, F, T = x.size()
        x = x.permute(0, 3, 1, 2).contiguous()  # (Batch, Time, Channels, Freq)
        x = x.view(B, T, C * F)                 # (Batch, T, 2048)
        output, hidden = self.gru(x)
        final_forward  = hidden[-2, :, :]
        final_backward = hidden[-1, :, :]
        final_state = torch.cat((final_forward, final_backward), dim=1)
        out = self.head(final_state)
        if out.shape[1] == 1:
            out = out.squeeze(1)
        return out

In [ ]:
# ── GRU Sequential Hyperparameter Search ────────────────────────────────────
# Strategy: for each architecture, sweep LR (5 runs), dropout (3 runs),
# hidden size (3 runs) — all others fixed at defaults.  11 runs × 8 archs = 88 total.
gru_results = []
runs_per_arch = len(LR_SEARCH) + len(DROPOUT_SEARCH) + len(GRU_SIZE_SEARCH)  # 11
total_runs    = len(ARCH_CONFIGS) * runs_per_arch
run_idx       = 0

for arch_i, arch in enumerate(ARCH_CONFIGS, 1):
    num_cnn = arch['num_cnn_layers']
    num_gru = arch['num_gru_layers']
    num_mlp = arch['num_mlp_layers']
    arch_tag = f"cnn{num_cnn}_gl{num_gru}_mlp{num_mlp}"

    # ── Phase 1: LR sweep (dropout=DEFAULT, hidden=DEFAULT) ──────────────
    for lr in LR_SEARCH:
        run_idx += 1
        dropout, gru_hidden = DEFAULT_DROPOUT, DEFAULT_HIDDEN
        print(f"\n{'='*70}")
        print(f"[GRU {run_idx}/{total_runs}]  Arch {arch_i}/{len(ARCH_CONFIGS)} ({arch_tag})  |  LR sweep")
        print(f"  lr={lr:.0e}  dropout={dropout}  gru_hidden={gru_hidden}  cnn={num_cnn}  gru_layers={num_gru}  mlp={num_mlp}")
        print(f"{'='*70}")

        ckpt = Path(f"/content/models/gru_{arch_tag}_lr{lr:.0e}_do{dropout}_hs{gru_hidden}.pt")
        model = GRUSpectrogramCNN(
            num_outputs=1, gru_hidden=gru_hidden, num_gru_layers=num_gru,
            num_cnn_layers=num_cnn, num_mlp_layers=num_mlp, dropout=dropout
        ).to(DEVICE)
        best_mae_bpm = train_model(
            model, train_loader, val_loader, bpm_std,
            epochs=N_EPOCHS, lr=lr, weight_decay=WEIGHT_DECAY,
            checkpoint_path=ckpt, device=DEVICE
        )
        gru_results.append({
            'lr': lr, 'dropout': dropout, 'gru_hidden': gru_hidden,
            'num_cnn_layers': num_cnn, 'num_gru_layers': num_gru, 'num_mlp_layers': num_mlp,
            'val_mae_bpm': best_mae_bpm, 'ckpt': ckpt
        })
        print(f"  → best val MAE = {best_mae_bpm:.2f} BPM")

    # ── Phase 2: Dropout sweep (lr=DEFAULT, hidden=DEFAULT) ──────────────
    for dropout in DROPOUT_SEARCH:
        run_idx += 1
        lr, gru_hidden = DEFAULT_LR, DEFAULT_HIDDEN
        print(f"\n{'='*70}")
        print(f"[GRU {run_idx}/{total_runs}]  Arch {arch_i}/{len(ARCH_CONFIGS)} ({arch_tag})  |  Dropout sweep")
        print(f"  lr={lr:.0e}  dropout={dropout}  gru_hidden={gru_hidden}  cnn={num_cnn}  gru_layers={num_gru}  mlp={num_mlp}")
        print(f"{'='*70}")

        ckpt = Path(f"/content/models/gru_{arch_tag}_lr{lr:.0e}_do{dropout}_hs{gru_hidden}.pt")
        model = GRUSpectrogramCNN(
            num_outputs=1, gru_hidden=gru_hidden, num_gru_layers=num_gru,
            num_cnn_layers=num_cnn, num_mlp_layers=num_mlp, dropout=dropout
        ).to(DEVICE)
        best_mae_bpm = train_model(
            model, train_loader, val_loader, bpm_std,
            epochs=N_EPOCHS, lr=lr, weight_decay=WEIGHT_DECAY,
            checkpoint_path=ckpt, device=DEVICE
        )
        gru_results.append({
            'lr': lr, 'dropout': dropout, 'gru_hidden': gru_hidden,
            'num_cnn_layers': num_cnn, 'num_gru_layers': num_gru, 'num_mlp_layers': num_mlp,
            'val_mae_bpm': best_mae_bpm, 'ckpt': ckpt
        })
        print(f"  → best val MAE = {best_mae_bpm:.2f} BPM")

    # ── Phase 3: Hidden size sweep (lr=DEFAULT, dropout=DEFAULT) ─────────
    for gru_hidden in GRU_SIZE_SEARCH:
        run_idx += 1
        lr, dropout = DEFAULT_LR, DEFAULT_DROPOUT
        print(f"\n{'='*70}")
        print(f"[GRU {run_idx}/{total_runs}]  Arch {arch_i}/{len(ARCH_CONFIGS)} ({arch_tag})  |  Hidden size sweep")
        print(f"  lr={lr:.0e}  dropout={dropout}  gru_hidden={gru_hidden}  cnn={num_cnn}  gru_layers={num_gru}  mlp={num_mlp}")
        print(f"{'='*70}")

        ckpt = Path(f"/content/models/gru_{arch_tag}_lr{lr:.0e}_do{dropout}_hs{gru_hidden}.pt")
        model = GRUSpectrogramCNN(
            num_outputs=1, gru_hidden=gru_hidden, num_gru_layers=num_gru,
            num_cnn_layers=num_cnn, num_mlp_layers=num_mlp, dropout=dropout
        ).to(DEVICE)
        best_mae_bpm = train_model(
            model, train_loader, val_loader, bpm_std,
            epochs=N_EPOCHS, lr=lr, weight_decay=WEIGHT_DECAY,
            checkpoint_path=ckpt, device=DEVICE
        )
        gru_results.append({
            'lr': lr, 'dropout': dropout, 'gru_hidden': gru_hidden,
            'num_cnn_layers': num_cnn, 'num_gru_layers': num_gru, 'num_mlp_layers': num_mlp,
            'val_mae_bpm': best_mae_bpm, 'ckpt': ckpt
        })
        print(f"  → best val MAE = {best_mae_bpm:.2f} BPM")

# ── Summary ───────────────────────────────────────────────────────────────────
gru_results_df = pd.DataFrame(gru_results).sort_values('val_mae_bpm').reset_index(drop=True)
print("\n=== GRU Hyperparameter Search Results (sorted by val MAE) ===")
print(gru_results_df[['lr', 'dropout', 'gru_hidden', 'num_cnn_layers', 'num_gru_layers', 'num_mlp_layers', 'val_mae_bpm']].to_string(index=False))

best_gru = gru_results_df.iloc[0]
shutil.copy(best_gru['ckpt'], MODEL02_SAVE_PATH)
print(f"\nBest GRU → lr={best_gru['lr']:.0e}, dropout={best_gru['dropout']}, "
      f"gru_hidden={int(best_gru['gru_hidden'])}, cnn={int(best_gru['num_cnn_layers'])}, "
      f"gru_layers={int(best_gru['num_gru_layers'])}, mlp={int(best_gru['num_mlp_layers'])}, "
      f"val MAE={best_gru['val_mae_bpm']:.2f} BPM")
print(f"Best checkpoint saved to {MODEL02_SAVE_PATH}")

# ── Save all results to CSV ──────────────────────────────────────────────────
csv_local  = SAVE / 'gru_search_results.csv'
csv_drive  = Path('/content/drive/MyDrive/6600_project/gru_search_results.csv')
gru_results_df.to_csv(csv_local, index=False)
shutil.copy(csv_local, csv_drive)
print(f"Results saved to {csv_drive}")




[GRU 1/88]  Arch 1/8 (cnn3_gl1_mlp1)  |  LR sweep
  lr=1e-03  dropout=0.3  gru_hidden=256  cnn=3  gru_layers=1  mlp=1
Epoch  1 | Loss: 2.0185 | Val MAE: 22.2 BPM ✓ saved
Epoch  2 | Loss: 0.8131 | Val MAE: 21.6 BPM ✓ saved
Epoch  3 | Loss: 0.7749 | Val MAE: 20.9 BPM ✓ saved
Epoch  4 | Loss: 0.7538 | Val MAE: 20.8 BPM ✓ saved
Epoch  5 | Loss: 0.7378 | Val MAE: 20.3 BPM ✓ saved
Epoch  6 | Loss: 0.7258 | Val MAE: 20.3 BPM (no improvement 1/3)
Epoch  7 | Loss: 0.7107 | Val MAE: 20.4 BPM (no improvement 2/3)
Epoch  8 | Loss: 0.6883 | Val MAE: 20.1 BPM ✓ saved
Epoch  9 | Loss: 0.6651 | Val MAE: 19.6 BPM ✓ saved
Epoch 10 | Loss: 0.6481 | Val MAE: 21.6 BPM (no improvement 1/3)
Total training time: 6.8 minutes
  → best val MAE = 19.58 BPM

[GRU 2/88]  Arch 1/8 (cnn3_gl1_mlp1)  |  LR sweep
  lr=5e-04  dropout=0.3  gru_hidden=256  cnn=3  gru_layers=1  mlp=1
Epoch  1 | Loss: 1.2085 | Val MAE: 22.0 BPM ✓ saved
Epoch  2 | Loss: 0.8022 | Val MAE: 22.0 BPM (no improvement 1/3)
Epoch  3 | Loss: 0.7669 

In [ ]:
# Test GRU Model
print("\n--- Testing CNN+GRU ---")
gru_model = GRUSpectrogramCNN(
    num_outputs=1,
    gru_hidden=int(best_gru['gru_hidden']),
    num_gru_layers=int(best_gru['num_gru_layers']),
    num_cnn_layers=int(best_gru['num_cnn_layers']),
    num_mlp_layers=int(best_gru['num_mlp_layers']),
    dropout=best_gru['dropout']
).to(DEVICE)
gru_preds, gru_actuals = test_model_and_plot(
    gru_model, test_loader, bpm_std, bpm_mean,
    checkpoint_path=MODEL02_SAVE_PATH, device=DEVICE
)